# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 146 (delta 53), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.89 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/Flyrank-assignment1


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The key numeric signals have noticeably different distributions and several show strong right-skew. For example, impressions_90d has a mean of 5,200 compared with a median of 731, while clicks_90d has a mean of 16.1 compared with a median of 1. avg_position also has a wide range from 0 to 245. These heavy tails mean that averages can be influenced by a relatively small number of high-value pages, so later analysis should consider distributions and robust comparisons rather than relying only on means. The five inspected fields have no missing values.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

numeric_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

print("=== Distribution summary ===")

display(
    df[numeric_cols].describe().T[
        ["count", "mean", "50%", "std", "min", "max"]
    ]
)

print("\n=== Missing values ===")

print(
    df[numeric_cols].isna().sum()
)

=== Distribution summary ===


,count,mean,50%,std,min,max
impressions_90d,30000.0,5200.366300,731.00,16838.019547,1.0,517715.0
clicks_90d,30000.0,16.097333,1.00,75.076958,0.0,4178.0
ctr,30000.0,0.510733,0.07,3.279162,0.0,100.0
avg_position,30000.0,16.342380,10.80,15.216790,0.0,245.0
engagement_rate,30000.0,2.534520,0.00,8.310096,0.0,100.0



=== Missing values ===
impressions_90d    0
clicks_90d         0
ctr                0
avg_position       0
engagement_rate    0
dtype: int64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 — CTR: I expect pages with lower CTR to show weaker search performance because fewer impressions result in clicks.

Signal 2 — Average position: I expect worse average positions to be associated directionally with weaker performance.

Signal 3 — Content age: I want to test whether older content is more likely to be declining.

Each signal is treated as a directional association rather than proof of causation. The verdicts below are based on the observed group differences.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


def signal_test(feature, low_label, high_label):
    median = df[feature].median()

    low = df[df[feature] <= median]["is_declining_label"]
    high = df[df[feature] > median]["is_declining_label"]

    low_rate = low.mean()
    high_rate = high.mean()

    difference = high_rate - low_rate

    if abs(difference) < 0.03:
        verdict = "MIXED"
    elif difference > 0:
        verdict = "CONFIRMED"
    else:
        verdict = "OPPOSITE"

    print(f"\n{feature}")
    print(f"{low_label}: {low_rate:.3f}")
    print(f"{high_label}: {high_rate:.3f}")
    print(f"Difference: {difference:+.3f}")
    print(f"Verdict: {verdict}")


signal_test(
    "ctr",
    "Lower CTR",
    "Higher CTR"
)

signal_test(
    "avg_position",
    "Better position",
    "Worse position"
)

signal_test(
    "content_age_days",
    "Newer content",
    "Older content"
)


ctr
Lower CTR: 0.524
Higher CTR: 0.561
Difference: +0.037
Verdict: CONFIRMED

avg_position
Better position: 0.521
Worse position: 0.564
Difference: +0.043
Verdict: CONFIRMED

content_age_days
Newer content: 0.621
Older content: 0.456
Difference: -0.164
Verdict: OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

I will test freshness_tier as a flag-linked signal. The assumption being tested is that content freshness may be associated with search-performance trends. I will compare the observed declining rate across freshness tiers. A difference between groups would support the rule directionally, but would not establish that freshness itself causes a change in performance.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          pages=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .sort_values("declining_rate", ascending=False)
)

print("=== Freshness tier vs declining rate ===")

display(flag_test)

print(
    "\nHighest observed declining rate:",
    flag_test["declining_rate"].idxmax()
)

print(
    "Lowest observed declining rate:",
    flag_test["declining_rate"].idxmin()
)

gap = (
    flag_test["declining_rate"].max()
    - flag_test["declining_rate"].min()
)

print(f"Declining-rate gap: {gap:.3f}")

=== Freshness tier vs declining rate ===


,pages,declining_rate
freshness_tier,,
91-180,9171,0.611057
31-90,175,0.588571
0-30,20480,0.511377
181+,174,0.471264



Highest observed declining rate: 91-180
Lowest observed declining rate: 181+
Declining-rate gap: 0.140


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit helps identify which observable characteristics are worth carrying into later modeling and which assumptions may be weak or mixed. The content team should use the stronger signals to prioritize pages for investigation rather than treating them as guaranteed causes of decline. The results are directional and should be validated on held-out clients before being used for operational decisions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Practical takeaway:")

print(
    "Use the strongest observed signals to prioritize "
    "pages for review."
)

print(
    "Treat all relationships as directional until "
    "validated on held-out clients."
)

print(
    "Do not interpret these associations as causal proof."
)

Practical takeaway:
Use the strongest observed signals to prioritize pages for review.
Treat all relationships as directional until validated on held-out clients.
Do not interpret these associations as causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.